# BRIGHT Evaluation - CPU-Safe Baseline

This notebook evaluates models on BRIGHT using a CPU-friendly approach that avoids:
- Compiled PyLate/PLAID extensions (architecture issues on ARM64)
- Large memory allocations (buffer errors)
- Complex distributed setup

## Methodology
Based on [NohTow's BRIGHT evaluation](https://gist.github.com/NohTow/3f27d2816b92d5c76f0e63aa7757cf4b) but adapted for CPU execution:
- Uses MTEB's BrightRetrieval task for proper test splits
- Encodes full corpus per domain (not training data)
- Respects excluded_ids filtering
- Small batch sizes (8-16) for memory safety
- Cosine similarity for retrieval scoring

## Models
- Dense baseline: `sentence-transformers/all-MiniLM-L6-v2`
- Target models: Any Sentence Transformer compatible model

---

## Google Colab Setup (Run this first if in Colab)

In [4]:
# %cd /Users/shishirjoshi/development/lab/reasoning_embedder/
!ls

reasoning_embedder  sample_data


In [2]:
%pip -q install pylate \
  voyager

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
reasoning-embedder 0.1.0 requires accelerate==1.10.1, but you have accelerate 1.12.0 which is incompatible.
reasoning-embedder 0.1.0 requires datasets==4.2.0, but you have datasets 4.0.0 which is incompatible.
reasoning-embedder 0.1.0 requires matplotlib==3.10.7, but you have matplotlib 3.10.0 which is incompatible.
reasoning-embedder 0.1.0 requires numpy==1.26.4, but you have numpy 2.0.2 which is incompatible.
reasoning-embedder 0.1.0 requires pandas==2.3.3, but you have pandas 2.2.2 which is incompatible.
reasoning-embedder 0.1.0 requires pylate==1.2.0, but you have pylate 1.3.4 which is incompatible.
reasoning-embedder 

In [8]:
%pip install transformers==4.41.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 16.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 32.8 MB/s  0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.15.2
    Uninstalling tokenizers-0.15.2:
      Successfully uninstalled tokenizers-0.15.2
  Attempting uninstall: transformers━━━━━━━━━━━━ 0/2 [tokenizers]
    Found existing installation: transformers 4.35.22m0/2 [tokenizers]
    Uninstalling transformers-4.35.2:╺━━━━━━━━━━━━━━━━━━━ 1/2 [transformers]
      Successfully uninstalled transformers-4.35.2━━━━━━━━━━━━ 1/2 [transformers]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers] [transformers]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
reasoning-embedder 0.1.0 requires accelerate==1.10.1, but you have accelerate 1.12.0 which is incompatible.
reasoning-em

In [ ]:
# ============================================================================
# GOOGLE COLAB SETUP
# This notebook is a *dense CPU* BRIGHT evaluation. It does NOT need `pylate`.
# Installing the full repo dependencies can trigger `pylate` conflicts on Colab.
# ============================================================================

import os

# Step 1: Clone the repository (optional; this notebook does not require it)
if not os.path.exists('/content/reasoning_embedder'):
    !git clone https://github.com/shishir-joshi/reasoning_embedder.git /content/reasoning_embedder
    print("✓ Repository cloned")
else:
    print("✓ Repository already exists")

# Step 2: Install only the packages needed for *this* notebook
# Use %pip so installs happen in the active kernel environment.
%pip -q install -U pip

%pip -q install \
  numpy \
  pandas \
  scipy \
  scikit-learn \
  matplotlib \
  seaborn \
  torch \
  transformers \
  sentence-transformers \
  datasets \
  accelerate \
  mteb \
  pylate \
  voyager

# Step 3 (optional): install the repo *without* dependencies
# This keeps CLI imports available without forcing `pylate`.
%cd /content/reasoning_embedder
%pip -q install -e . --no-deps

print("✓ Colab environment ready for dense BRIGHT evaluation")

✓ Repository already exists
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Operation cancelled by user
/content/reasoning_embedder


  Installing build dependencies ... canceled
ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 107, in _run_wrapper
    status = _inner_run()
             ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 98, in _inner_run
    return self.run(options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 85, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 388, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 79, in resolve
    collected = self.factory.collect_root_requirements(root_reqs)
    

In [5]:
# Sanity check: show versions and detect dependency conflicts

import sys
import subprocess

print("Python:", sys.version)

for pkg in ["torch", "transformers", "sentence-transformers", "datasets", "accelerate", "mteb", "pylate"]:
    try:
        out = subprocess.check_output([sys.executable, "-m", "pip", "show", pkg], text=True)
        ver_line = next((ln for ln in out.splitlines() if ln.startswith("Version:")), "Version: ?")
        print(f"{pkg}: {ver_line.split(':', 1)[1].strip()}")
    except subprocess.CalledProcessError:
        print(f"{pkg}: (not installed)")

print("\nRunning `pip check`...")
try:
    check_out = subprocess.check_output([sys.executable, "-m", "pip", "check"], text=True)
    print(check_out.strip() or "No broken requirements found.")
except subprocess.CalledProcessError as e:
    # pip check exits non-zero on conflicts
    print(e.output.strip() or str(e))

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
torch: 2.8.0
transformers: 4.56.2
sentence-transformers: 5.1.1
datasets: 4.0.0
accelerate: 1.12.0
mteb: 2.5.4
pylate: 1.3.4

Running `pip check`...
ipython 7.34.0 requires jedi, which is not installed.
reasoning-embedder 0.1.0 has requirement accelerate==1.10.1, but you have accelerate 1.12.0.
reasoning-embedder 0.1.0 has requirement datasets==4.2.0, but you have datasets 4.0.0.
reasoning-embedder 0.1.0 has requirement matplotlib==3.10.7, but you have matplotlib 3.10.0.
reasoning-embedder 0.1.0 has requirement numpy==1.26.4, but you have numpy 2.0.2.
reasoning-embedder 0.1.0 has requirement pandas==2.3.3, but you have pandas 2.2.2.
reasoning-embedder 0.1.0 has requirement pylate==1.2.0, but you have pylate 1.3.4.
reasoning-embedder 0.1.0 has requirement scikit-learn==1.7.2, but you have scikit-learn 1.6.1.
reasoning-embedder 0.1.0 has requirement scipy==1.16.2, but you have scipy 1.16.3.
reasoning-embedder 0.1.0 has requirement

## 1. Setup and Imports

In [1]:
import os
import json
import time
from pathlib import Path
from typing import Dict, List, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
import mteb

# Configuration - works for both local and Colab
# In Colab, current dir is /content/reasoning_embedder after setup
# Locally, adjust the path as needed
OUTPUT_DIR = "data/bright_evaluation_results/cpu_baseline"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Enable CPU-safe settings
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"

print(f"✓ Imports complete")
print(f"✓ Working directory: {os.getcwd()}")
print(f"✓ Output directory: {OUTPUT_DIR}")

RuntimeError: Failed to import transformers.trainer because of the following error (look up to see its traceback):
cannot import name 'EncoderDecoderCache' from 'transformers' (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)

## 2. Load BRIGHT Dataset via MTEB

MTEB provides the proper BRIGHT splits with:
- Test queries (not training data)
- Full corpus per domain
- Query-document relevance judgments (qrels)
- Excluded documents that should be filtered

In [ ]:
# Load BRIGHT retrieval task
tasks = mteb.get_tasks(tasks=["BrightRetrieval"])
task = tasks[0]

# Load data (this downloads if not cached)
print("Loading BRIGHT data...")
task.load_data()

# Inspect available splits
print(f"\nAvailable evaluation sets: {list(task.queries.keys())}")

# Show stats for each domain
for eval_set in task.queries.keys():
    n_queries = len(task.queries[eval_set]["standard"])
    n_docs = len(task.corpus[eval_set]["standard"])
    n_qrels = len(task.relevant_docs[eval_set]["standard"])
    print(f"\n{eval_set}:")
    print(f"  Queries: {n_queries}")
    print(f"  Documents: {n_docs}")
    print(f"  Qrels (query-doc pairs): {n_qrels}")

## 3. CPU-Safe Retrieval Functions

Key strategies:
- **Small batches**: 8-16 documents at a time to avoid buffer errors
- **Numpy arrays**: Convert to numpy for efficient cosine similarity
- **Chunked processing**: Process documents in chunks to manage memory
- **No CUDA/compiled extensions**: Pure Python/numpy operations

In [ ]:
def encode_texts_safe(model: SentenceTransformer, texts: List[str], batch_size: int = 8, show_progress: bool = True) -> np.ndarray:
    """
    Encode texts with small batch sizes for CPU safety.
    
    Args:
        model: Sentence transformer model
        texts: List of texts to encode
        batch_size: Small batch size (8-16 recommended for CPU)
        show_progress: Whether to print progress
    
    Returns:
        Numpy array of embeddings [num_texts, embedding_dim]
    """
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        show_progress_bar=show_progress,
        normalize_embeddings=False  # We'll normalize during similarity computation
    )
    return embeddings


def cosine_similarity_search(query_embs: np.ndarray, doc_embs: np.ndarray, k: int = 100) -> List[List[Dict]]:
    """
    Compute cosine similarity and return top-k documents for each query.
    
    Args:
        query_embs: Query embeddings [num_queries, dim]
        doc_embs: Document embeddings [num_docs, dim]
        k: Number of top documents to retrieve
    
    Returns:
        List of top-k results per query, each result is a dict with 'id' and 'score'
    """
    # Normalize embeddings for cosine similarity
    query_norm = query_embs / (np.linalg.norm(query_embs, axis=1, keepdims=True) + 1e-9)
    doc_norm = doc_embs / (np.linalg.norm(doc_embs, axis=1, keepdims=True) + 1e-9)
    
    # Compute similarity matrix: [num_queries, num_docs]
    similarity_matrix = query_norm @ doc_norm.T
    
    # Get top-k for each query
    results = []
    for i, query_scores in enumerate(similarity_matrix):
        # Get indices of top-k scores (descending order)
        top_k_indices = np.argsort(query_scores)[::-1][:k]
        
        # Create result list with doc indices and scores
        query_results = [
            {"id": int(idx), "score": float(query_scores[idx])}
            for idx in top_k_indices
        ]
        results.append(query_results)
    
    return results


def filter_excluded_docs(scores: List[List[Dict]], doc_ids: List[str], excluded_docs: Dict[str, List[str]]) -> List[List[Dict]]:
    """
    Filter out excluded documents from retrieval results.
    
    Args:
        scores: List of top-k results per query
        doc_ids: List mapping array indices to document IDs
        excluded_docs: Dict mapping query IDs to lists of excluded doc IDs
    
    Returns:
        Filtered scores with excluded documents removed
    """
    query_ids = list(excluded_docs.keys())
    filtered_scores = []
    
    for query_idx, (query_scores, query_id) in enumerate(zip(scores, query_ids)):
        excluded_ids = excluded_docs.get(query_id, [])
        
        # If no exclusions or N/A, keep all
        if excluded_ids == "N/A" or not excluded_ids:
            filtered_scores.append(query_scores)
            continue
        
        # Filter out excluded documents
        filtered_query_scores = []
        for result in query_scores:
            doc_id = doc_ids[result["id"]]
            if doc_id not in excluded_ids:
                # Update the ID to be the actual document ID, not array index
                filtered_query_scores.append({"id": doc_id, "score": result["score"]})
        
        filtered_scores.append(filtered_query_scores)
    
    return filtered_scores


def compute_retrieval_metrics(scores: List[List[Dict]], qrels: Dict[str, Dict[str, int]], query_ids: List[str]) -> Dict[str, float]:
    """
    Compute standard retrieval metrics: Recall@k, NDCG@k, MAP.
    
    Args:
        scores: List of ranked results per query (list of dicts with 'id' and 'score')
        qrels: Query relevance judgments {query_id: {doc_id: relevance_score}}
        query_ids: List of query IDs in order
    
    Returns:
        Dictionary of metric scores
    """
    from mteb.evaluation.evaluators import RetrievalEvaluator
    
    # Convert scores to MTEB format: {query_id: {doc_id: score}}
    results = {}
    for query_id, query_scores in zip(query_ids, scores):
        results[query_id] = {
            result["id"]: result["score"]
            for result in query_scores
        }
    
    # Use MTEB's evaluator
    evaluator = RetrievalEvaluator()
    metrics = evaluator(qrels, results)
    
    return metrics


print("CPU-safe retrieval functions defined!")

## 4. Evaluate a Single Model on BRIGHT

This cell evaluates one model across all BRIGHT domains.

In [ ]:
def evaluate_model_on_bright(
    model_name: str,
    batch_size: int = 8,
    k: int = 100,
    device: str = "cpu"
) -> Dict[str, Dict[str, float]]:
    """
    Evaluate a single model on all BRIGHT domains.
    
    Args:
        model_name: HuggingFace model identifier
        batch_size: Encoding batch size (keep small for CPU)
        k: Number of documents to retrieve
        device: Device to use ("cpu", "cuda", "mps")
    
    Returns:
        Dictionary mapping domain names to metric dictionaries
    """
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*60}\n")
    
    # Load model
    print(f"Loading model on {device}...")
    model = SentenceTransformer(model_name, device=device)
    
    all_results = {}
    
    # Evaluate on each domain
    for eval_set in task.queries.keys():
        print(f"\n--- Evaluating on {eval_set} ---")
        start_time = time.time()
        
        try:
            # Get data
            corpus = task.corpus[eval_set]["standard"]
            queries = task.queries[eval_set]["standard"]
            qrels = task.relevant_docs[eval_set]["standard"]
            
            # Handle excluded docs
            if "excluded" in task.relevant_docs[eval_set]:
                excluded_docs = task.relevant_docs[eval_set]["excluded"]
            else:
                excluded_docs = {qid: "N/A" for qid in queries.keys()}
            
            # Prepare texts
            doc_ids = list(corpus.keys())
            doc_texts = [corpus[doc_id] for doc_id in doc_ids]
            
            query_ids = list(queries.keys())
            query_texts = [queries[qid] for qid in query_ids]
            
            print(f"  Documents: {len(doc_texts)}")
            print(f"  Queries: {len(query_texts)}")
            
            # Encode documents
            print(f"  Encoding documents (batch_size={batch_size})...")
            doc_embs = encode_texts_safe(model, doc_texts, batch_size=batch_size, show_progress=True)
            print(f"  Document embeddings shape: {doc_embs.shape}")
            
            # Encode queries
            print(f"  Encoding queries...")
            query_embs = encode_texts_safe(model, query_texts, batch_size=batch_size, show_progress=True)
            print(f"  Query embeddings shape: {query_embs.shape}")
            
            # Retrieve
            print(f"  Computing similarity and retrieving top-{k}...")
            scores = cosine_similarity_search(query_embs, doc_embs, k=k)
            
            # Filter excluded documents
            print(f"  Filtering excluded documents...")
            filtered_scores = filter_excluded_docs(scores, doc_ids, excluded_docs)
            
            # Compute metrics
            print(f"  Computing metrics...")
            metrics = compute_retrieval_metrics(filtered_scores, qrels, query_ids)
            
            all_results[eval_set] = metrics
            
            elapsed = time.time() - start_time
            print(f"  ✓ Completed in {elapsed:.1f}s")
            print(f"  NDCG@10: {metrics.get('ndcg_at_10', 0.0):.4f}")
            print(f"  Recall@10: {metrics.get('recall_at_10', 0.0):.4f}")
            
        except Exception as e:
            print(f"  ✗ Error: {str(e)}")
            import traceback
            traceback.print_exc()
            all_results[eval_set] = {"error": str(e)}
    
    # Save results
    model_output_dir = Path(OUTPUT_DIR) / model_name.replace("/", "_")
    model_output_dir.mkdir(parents=True, exist_ok=True)
    
    results_file = model_output_dir / "bright_results.json"
    with open(results_file, "w") as f:
        json.dump(all_results, f, indent=2)
    
    print(f"\n✓ Results saved to {results_file}")
    
    return all_results


print("Evaluation function defined!")

## 5. Run Evaluation

Evaluate baseline model. Adjust batch_size based on your system:
- **ARM64/Apple Silicon**: Start with batch_size=8
- **x86_64 with more RAM**: Can try batch_size=16-32
- **If you get buffer errors**: Reduce batch_size to 4

In [ ]:
# Baseline model
baseline_results = evaluate_model_on_bright(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    batch_size=8,  # Small batch for CPU safety
    k=100,
    device="cpu"
)

## 6. Results Summary

Display results in a clean table format.

In [ ]:
def display_results(results: Dict[str, Dict[str, float]], model_name: str):
    """
    Display results in a formatted table.
    """
    print(f"\n{'='*80}")
    print(f"Results for: {model_name}")
    print(f"{'='*80}\n")
    
    rows = []
    for domain, metrics in results.items():
        if "error" in metrics:
            rows.append({
                "Domain": domain,
                "Status": "ERROR",
                "NDCG@10": "-",
                "Recall@10": "-",
                "MAP": "-"
            })
        else:
            rows.append({
                "Domain": domain,
                "Status": "✓",
                "NDCG@10": f"{metrics.get('ndcg_at_10', 0.0):.4f}",
                "Recall@10": f"{metrics.get('recall_at_10', 0.0):.4f}",
                "MAP": f"{metrics.get('map', 0.0):.4f}"
            })
    
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    
    # Compute average (excluding errors)
    valid_results = [m for m in results.values() if "error" not in m]
    if valid_results:
        avg_ndcg = np.mean([m.get('ndcg_at_10', 0.0) for m in valid_results])
        avg_recall = np.mean([m.get('recall_at_10', 0.0) for m in valid_results])
        avg_map = np.mean([m.get('map', 0.0) for m in valid_results])
        
        print(f"\n{'='*80}")
        print(f"Average across {len(valid_results)} domains:")
        print(f"  NDCG@10:   {avg_ndcg:.4f}")
        print(f"  Recall@10: {avg_recall:.4f}")
        print(f"  MAP:       {avg_map:.4f}")
        print(f"{'='*80}\n")

# Display baseline results
display_results(baseline_results, "sentence-transformers/all-MiniLM-L6-v2")

## 7. Compare Multiple Models (Optional)

Evaluate additional models for comparison.

In [ ]:
# Example: Evaluate another model
# Uncomment to run

# model2_results = evaluate_model_on_bright(
#     model_name="BAAI/bge-small-en-v1.5",
#     batch_size=8,
#     k=100,
#     device="cpu"
# )
# display_results(model2_results, "BAAI/bge-small-en-v1.5")

## 8. Visualize Comparison

Create comparison charts if multiple models are evaluated.

In [ ]:
# Load all model results from OUTPUT_DIR
def load_all_results() -> Dict[str, Dict[str, Dict[str, float]]]:
    """
    Load all saved model results.
    
    Returns:
        Dict mapping model names to their domain results
    """
    all_model_results = {}
    
    output_path = Path(OUTPUT_DIR)
    for model_dir in output_path.iterdir():
        if model_dir.is_dir():
            results_file = model_dir / "bright_results.json"
            if results_file.exists():
                with open(results_file, "r") as f:
                    results = json.load(f)
                model_name = model_dir.name.replace("_", "/")
                all_model_results[model_name] = results
    
    return all_model_results


def plot_comparison(all_results: Dict[str, Dict[str, Dict[str, float]]]):
    """
    Create comparison plots across models and domains.
    """
    if len(all_results) < 2:
        print("Need at least 2 models to compare. Run more evaluations first.")
        return
    
    # Prepare data for plotting
    plot_data = []
    for model_name, domain_results in all_results.items():
        for domain, metrics in domain_results.items():
            if "error" not in metrics:
                plot_data.append({
                    "Model": model_name,
                    "Domain": domain,
                    "NDCG@10": metrics.get("ndcg_at_10", 0.0),
                    "Recall@10": metrics.get("recall_at_10", 0.0)
                })
    
    df = pd.DataFrame(plot_data)
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # NDCG@10
    sns.barplot(data=df, x="Domain", y="NDCG@10", hue="Model", ax=axes[0])
    axes[0].set_title("NDCG@10 by Domain")
    axes[0].set_ylabel("NDCG@10")
    axes[0].tick_params(axis='x', rotation=45)
    
    # Recall@10
    sns.barplot(data=df, x="Domain", y="Recall@10", hue="Model", ax=axes[1])
    axes[1].set_title("Recall@10 by Domain")
    axes[1].set_ylabel("Recall@10")
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig(Path(OUTPUT_DIR) / "model_comparison.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"Comparison plot saved to {OUTPUT_DIR}/model_comparison.png")


# Load and plot
all_saved_results = load_all_results()
plot_comparison(all_saved_results)

## Notes

### Why This Approach Works
1. **Proper Test Split**: Uses MTEB's curated test queries, not training data
2. **Full Corpus**: Encodes all documents in each domain, not a small subset
3. **CPU-Safe**: Small batches and numpy operations avoid memory issues
4. **No Compiled Dependencies**: Pure Python/numpy/PyTorch, no JIT compilation

### Limitations
- Slower than GPU-accelerated methods
- Dense models only (no late-interaction/ColBERT without compiled extensions)
- May take 10-30 minutes per model depending on system

### Extending to ColBERT Models
To evaluate ColBERT models on CPU:
1. Use Voyager instead of PLAID for indexing (pure Python, no compilation)
2. Implement MaxSim scoring manually in numpy
3. Process queries one at a time to manage memory

See `explore_prepared_dataset.ipynb` for a Voyager-based ColBERT example.